# Modul 1 - Praktikum D2

Notebook ini mengerjakan tugas individu Modul 1 bagian D2: membaca dan menampilkan image, melihat ukuran image, mengakses pixel, membuat garis horizontal/vertikal/diagonal, serta membuat kotak dari kumpulan pixel putih.

## Jawaban Pertanyaan Praktikum D2

1. Eksekusi Python dilakukan menggunakan Google Colab karena Colab menyediakan environment siap pakai di browser, mendukung notebook interaktif, mudah dihubungkan dengan Google Drive/GitHub, dan tidak mengharuskan mahasiswa menginstal semua library di laptop masing-masing.

2. Kegunaan library pada praktikum: `numpy` untuk array/matriks citra, `pandas` untuk pengolahan data tabular jika diperlukan, `cv2`/OpenCV untuk membaca dan memproses citra, `skimage` untuk membaca citra dari URL dan fungsi image processing tambahan, `PIL` untuk manipulasi image berbasis Pillow, dan `matplotlib` untuk visualisasi citra/grafik. Tidak semua library wajib dipakai pada sesi ini; library dipakai sesuai kebutuhan kode.

3. Potongan kode konversi warna seperti `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` digunakan karena OpenCV membaca citra dalam urutan channel BGR, sedangkan Matplotlib menampilkan citra dalam urutan RGB. Jika tidak dikonversi, warna yang tampil bisa tertukar, misalnya merah terlihat kebiruan.

4. Nilai `[255, 255, 255]` adalah nilai warna putih pada citra 3 channel 8-bit. Setiap angka mewakili intensitas channel warna, dengan 0 paling gelap dan 255 paling terang.

5. Pixel adalah elemen terkecil penyusun gambar digital. Resolusi menunjukkan jumlah pixel pada gambar, misalnya lebar x tinggi. Semakin tinggi resolusi, semakin banyak pixel yang menyimpan detail sehingga gambar lebih tajam, tetapi ukuran data/file dan kebutuhan komputasinya juga lebih besar.

## Import Library

In [ ]:
import urllib.request
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from skimage import io
try:
    from google.colab.patches import cv2_imshow
except Exception:
    def cv2_imshow(image):
        if image.ndim == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(6, 4))
        plt.imshow(image, cmap='gray' if image.ndim == 2 else None)
        plt.axis('off')
        plt.show()
plt.rcParams['figure.dpi'] = 110

## Praktikum D2 Langkah 1-3: Membaca, Resize, Konversi RGB, dan Ukuran Image

In [ ]:
urls = [
    'https://iiif.lib.ncsu.edu/iiif/0052574/full/800,/0/default.jpg',
    'https://iiif.lib.ncsu.edu/iiif/0016007/full/800,/0/default.jpg',
    'https://placekitten.com/800/571',
]
def make_fallback_image(width=800, height=571):
    y, x = np.mgrid[0:height, 0:width]
    r = (x / width * 255).astype(np.uint8)
    g = (y / height * 255).astype(np.uint8)
    b = (((x + y) / (width + height)) * 255).astype(np.uint8)
    return np.dstack([r, g, b])
def read_rgb_from_url(url):
    try:
        img_rgb = io.imread(url)
        if img_rgb.ndim == 2:
            img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_GRAY2RGB)
        if img_rgb.shape[-1] == 4:
            img_rgb = img_rgb[:, :, :3]
        return img_rgb
    except Exception as exc:
        print(f'Gagal membaca {url}: {exc}. Menggunakan fallback image.')
        return make_fallback_image()
images_rgb = [read_rgb_from_url(url) for url in urls]
for index, img_rgb in enumerate(images_rgb, start=1):
    h, w = img_rgb.shape[:2]
    resized = cv2.resize(img_rgb, (w // 2, h // 2))
    combined = np.hstack([cv2.resize(img_rgb, (resized.shape[1], resized.shape[0])), resized])
    plt.figure(figsize=(9, 4))
    plt.imshow(combined)
    plt.title(f'Image {index}: asli disetarakan | resize 1/2')
    plt.axis('off')
    plt.show()
    print(f'Image {index}: shape={img_rgb.shape}, tinggi={h}, lebar={w}, channel={img_rgb.shape[2]}')

## Praktikum D2 Langkah 4 dan Tugas 2-4: Akses Pixel, Garis, dan Kotak

In [ ]:
img_tugas = images_rgb[0].copy()
tinggi, lebar = img_tugas.shape[:2]
# Garis horizontal putih di tengah gambar dengan panjang tertentu.
y_tengah = tinggi // 2
x_awal = lebar // 4
x_akhir = 3 * lebar // 4
img_tugas[y_tengah - 2:y_tengah + 3, x_awal:x_akhir] = [255, 255, 255]
# Garis vertikal putih.
x_vertikal = lebar // 2
img_tugas[tinggi // 5:4 * tinggi // 5, x_vertikal - 2:x_vertikal + 3] = [255, 255, 255]
# Garis menyilang diagonal putih.
for offset in range(-2, 3):
    for x in range(lebar):
        y = int((tinggi - 1) * x / (lebar - 1)) + offset
        if 0 <= y < tinggi:
            img_tugas[y, x] = [255, 255, 255]
# Kotak dari kumpulan pixel putih di sembarang tempat.
kotak_y1, kotak_y2 = tinggi // 6, tinggi // 6 + 90
kotak_x1, kotak_x2 = lebar // 10, lebar // 10 + 130
img_tugas[kotak_y1:kotak_y2, kotak_x1:kotak_x2] = [255, 255, 255]
plt.figure(figsize=(8, 5))
plt.imshow(img_tugas)
plt.title('Tugas D2: garis horizontal, vertikal, diagonal, dan kotak putih')
plt.axis('off')
plt.show()